# SignaCulture: T5-base on the frozen LSFB splits
This companion reuses the splits already saved by the mT5 notebook or the SSH run. It does not re-extract or resplit data. Select a T4 GPU, run in order, and authorize Drive. Target is Belgian LSFB; weak corpus alignment is not linguistic validation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys,json
DRIVE=Path('/content/drive/MyDrive/SignaCulture/mt5-lsfb-poc')
assert (DRIVE/'splits/split_report.json').is_file(), 'Prepare the frozen splits with the mT5 notebook first.'
WORK=Path('/content/signaculture-lsfb');WORK.mkdir(exist_ok=True)
TRAIN_STEPS=300
BASE_MODEL='google-t5/t5-base'  # Or: google/flan-t5-base
MODEL_REVISION={'google-t5/t5-base':'a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1','google/flan-t5-base':'7bcac572ce56db69c1ea7c8af255c5d7c9672fc2'}[BASE_MODEL]
RUN=DRIVE/(BASE_MODEL.rsplit('/',1)[-1]+f'-steps-{TRAIN_STEPS}')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','accelerate==1.15.0','sentencepiece==0.2.3','huggingface-hub==0.36.2'],check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print(torch.cuda.get_device_name())


In [ ]:
SCRIPT=WORK/"lsfb_poc.py"
SCRIPT.write_text('"""Small, recording/signer-disjoint French → Belgian LSFB experiment."""\nimport argparse, csv, hashlib, json, math, random, tarfile\nfrom pathlib import Path\n\nPREFIX = \'translate French to LSFB glosses: \'\nREPO = \'ppoitier/lsfb-cont\'\n\ndef dump(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n\ndef extract(archive, output, limit=300, overlap=.6):\n    recordings = {}\n    with tarfile.open(archive) as tar:\n        for member in tar:\n            suffix = next((s for s in [\'annotations.translation.json\', \'annotations.both_hands.json\', \'signer.txt\'] if member.name.endswith(\'.\'+s)), None)\n            if not suffix or not member.isfile(): continue\n            if member.size > 20_000_000: raise ValueError(\'Unexpectedly large annotation\')\n            key = member.name[:-(len(suffix)+1)]\n            value = tar.extractfile(member).read().decode(\'utf-8\')\n            recordings.setdefault(key, {})[suffix] = value.strip() if suffix==\'signer.txt\' else json.loads(value)\n    rows=[]; seen=set()\n    for recording, data in sorted(recordings.items()):\n        translations=data.get(\'annotations.translation.json\',[])\n        signs=data.get(\'annotations.both_hands.json\',[])\n        if not isinstance(translations,list) or not isinstance(signs,list): raise ValueError(\'Unrecognized corpus JSON: expected segment lists\')\n        for t in translations:\n            start,end=float(t[\'start_ms\']),float(t[\'end_ms\'])\n            if end<=start or not t.get(\'text\',\'\').strip(): continue\n            events=set()\n            for s in signs:\n                a,b=float(s[\'start_ms\']),float(s[\'end_ms\']); gloss=s.get(\'gloss\')\n                if b>a and isinstance(gloss,str) and gloss.strip() and max(0,min(end,b)-max(start,a))/(b-a)>=overlap:\n                    events.add((a,b,gloss.strip()))\n            target=\' \'.join(g for a,b,g in sorted(events))\n            text=\' \'.join(t[\'text\'].split())\n            # Exact repeated French text is removed globally, even when targets differ.\n            if not target or text.casefold() in seen: continue\n            seen.add(text.casefold())\n            rows.append(dict(id=f\'{recording}:{start}:{end}\',recording_id=recording,signer_id=data.get(\'signer.txt\'),language=\'LSFB\',french_text=text,gloss_sequence=target,source=REPO,annotation_tier=\'both_hands\',review_status=\'unreviewed_auto_alignment\',start_ms=start,end_ms=end))\n    # Sample across recordings before capping; never take only the first recording.\n    random.Random(42).shuffle(rows); rows=rows[:limit]\n    Path(output).parent.mkdir(parents=True,exist_ok=True)\n    Path(output).write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows))\n    report=dict(candidates=len(rows),recordings=len({r[\'recording_id\'] for r in rows}),signers=sorted({r[\'signer_id\'] for r in rows if r[\'signer_id\']}),archive_sha256=hashlib.file_digest(open(archive,\'rb\'),\'sha256\').hexdigest(),alignment=\'weak temporal overlap; simultaneous signs flattened\',license=\'CC-BY-NC-SA-4.0\')\n    dump(str(output)+\'.audit.json\',report); return report\n\ndef split(source, folder, allow_weak=False):\n    rows=[json.loads(l) for l in Path(source).read_text().splitlines() if l.strip()]\n    rows=[r for r in rows if r[\'review_status\']==\'approved\' or allow_weak and r[\'review_status\']==\'unreviewed_auto_alignment\']\n    if not rows: raise ValueError(\'No eligible examples. Review candidates or explicitly enable weak experimental labels.\')\n    signer_groups={r.get(\'signer_id\') for r in rows}\n    group=\'signer_id\' if None not in signer_groups and len(signer_groups)>=3 else \'recording_id\'\n    groups=sorted({r[group] for r in rows});random.Random(42).shuffle(groups)\n    if len(groups)<3: raise ValueError(\'Fewer than three independent groups: obtain another training shard\')\n    n=max(1,round(.15*len(groups))); selected={\'test\':set(groups[:n]),\'val\':set(groups[n:2*n]),\'train\':set(groups[2*n:])}\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);report={\'grouping\':group,\'weak_supervision\':allow_weak,\'evaluation_scope\':\'internal held-out subset of official training shard, not official benchmark\',\'splits\':{}}\n    for name, ids in selected.items():\n        part=[r for r in rows if r[group] in ids]\n        if not part: raise ValueError(\'Empty split\')\n        path=folder/(name+\'.jsonl\');path.write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in part))\n        report[\'splits\'][name]={\'examples\':len(part),\'groups\':sorted(ids),\'sha256\':hashlib.sha256(path.read_bytes()).hexdigest()}\n    dump(folder/\'split_report.json\',report);return report\n\ndef distance(a,b):\n    prev=list(range(len(b)+1))\n    for i,x in enumerate(a,1):\n        cur=[i]\n        for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n        prev=cur\n    return prev[-1]\n\ndef train(data, output, steps=30, base=\'google/mt5-small\', revision=None, max_length=96, eval_every=0):\n    import torch\n    from torch.utils.data import Dataset\n    from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments,TrainerCallback,set_seed\n    set_seed(42);data=Path(data);output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    report=json.loads((data/\'split_report.json\').read_text())\n    for name, entry in report[\'splits\'].items():\n        if hashlib.sha256((data/(name+\'.jsonl\')).read_bytes()).hexdigest()!=entry[\'sha256\']:raise ValueError(\'Split changed since freezing\')\n    fingerprint=hashlib.sha256((data/\'split_report.json\').read_bytes()).hexdigest()\n    manifest={\'base\':base,\'revision\':revision,\'split_sha256\':fingerprint,\'steps\':steps,\'language\':\'LSFB\',\'weak_supervision\':report[\'weak_supervision\'],\'precision\':\'bf16 when supported; otherwise fp32, never fp16\'}\n    if max_length!=96:manifest[\'max_length\']=max_length\n    if eval_every:manifest[\'eval_every\']=eval_every\n    plan=output/\'run_config.json\'\n    if plan.exists() and json.loads(plan.read_text())!=manifest:raise ValueError(\'Different configuration: use a new Drive run folder\')\n    dump(plan,manifest)\n    if (output/\'COMPLETED.json\').is_file() and all((output/name).is_file() for name in [\'final/model.safetensors\',\'final/config.json\',\'final/tokenizer_config.json\',\'metrics.json\']):\n        print(\'Run already complete; preserving the saved model and results.\',flush=True)\n        return\n    tok=AutoTokenizer.from_pretrained(base,revision=revision,use_fast=False)\n    model=AutoModelForSeq2SeqLM.from_pretrained(base,revision=revision);model.config.use_cache=False\n    class Pairs(Dataset):\n        def __init__(self,name):\n            self.rows=[json.loads(l) for l in (data/(name+\'.jsonl\')).read_text().splitlines()]\n            self.items=[]\n            for r in self.rows:\n                item=tok(PREFIX+r[\'french_text\'],return_token_type_ids=False);item[\'labels\']=tok(text_target=r[\'gloss_sequence\'])[\'input_ids\']\n                if max(len(item[\'input_ids\']),len(item[\'labels\']))>max_length: raise ValueError(\'Overlength example: review or segment it; no silent target truncation\')\n                self.items.append(item)\n        def __len__(self): return len(self.items)\n        def __getitem__(self,i): return self.items[i]\n    class Finite(TrainerCallback):\n        def on_log(self,args,state,control,logs=None,**kwargs):\n            if any(isinstance(v,(float,int)) and not math.isfinite(v) for v in (logs or {}).values()): raise ValueError(\'Nonfinite training log\')\n    interval=eval_every or max(1,steps//2)\n    args=Seq2SeqTrainingArguments(output_dir=str(output/\'checkpoints\'),max_steps=steps,per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=4,learning_rate=5e-5,optim=\'adafactor\',gradient_checkpointing=True,fp16=False,bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),eval_strategy=\'steps\',eval_steps=interval,save_steps=interval,save_total_limit=2,load_best_model_at_end=True,metric_for_best_model=\'eval_loss\',logging_steps=1,logging_nan_inf_filter=False,report_to=\'none\',seed=42)\n    trainer=Seq2SeqTrainer(model=model,args=args,train_dataset=Pairs(\'train\'),eval_dataset=Pairs(\'val\'),data_collator=DataCollatorForSeq2Seq(tok,model=model),processing_class=tok,callbacks=[Finite()])\n    # A disconnect can leave a half-written directory on Drive. Resume only complete saves.\n    checkpoints=[]\n    for checkpoint in (output/\'checkpoints\').glob(\'checkpoint-*\'):\n        if all((checkpoint/name).is_file() for name in [\'trainer_state.json\',\'optimizer.pt\',\'scheduler.pt\']) and ((checkpoint/\'model.safetensors\').is_file() or (checkpoint/\'model.safetensors.index.json\').is_file()):\n            checkpoints.append(checkpoint)\n    latest=str(max(checkpoints,key=lambda p:int(p.name.split(\'-\')[-1]))) if checkpoints else None\n    trainer.train(resume_from_checkpoint=latest)\n    trainer.save_model(str(output/\'final\'));tok.save_pretrained(output/\'final\');dump(output/\'history.json\',trainer.state.log_history)\n    # Test is decoded only after validation checkpoint selection.\n    from difflib import SequenceMatcher\n    model.eval();model.config.use_cache=True;rows=Pairs(\'test\').rows;training=Pairs(\'train\').rows;predictions=[];refs=[];baseline=[]\n    torch.set_num_threads(2)\n    for begin in range(0,len(rows),8):\n        part=rows[begin:begin+8]\n        with torch.inference_mode():\n            x=tok([PREFIX+r[\'french_text\'] for r in part],padding=True,return_tensors=\'pt\',return_token_type_ids=False).to(model.device)\n            outputs=model.generate(**x,max_new_tokens=max_length,num_beams=4)\n        for r,prediction in zip(part,tok.batch_decode(outputs,skip_special_tokens=True)):\n            predictions.append({\'french\':r[\'french_text\'],\'reference\':r[\'gloss_sequence\'],\'prediction\':prediction,\'recording_id\':r[\'recording_id\']});refs.append(r[\'gloss_sequence\'])\n            baseline.append(max(training,key=lambda t:SequenceMatcher(None,t[\'french_text\'].casefold(),r[\'french_text\'].casefold()).ratio())[\'gloss_sequence\'])\n        dump(output/\'predictions.partial.json\',predictions)\n        print(f\'Evaluated {len(predictions)}/{len(rows)} held-out examples\',flush=True)\n    def score(pred):return {\'exact_match\':sum(a==b for a,b in zip(refs,pred))/len(refs),\'wer\':sum(distance(a.split(),b.split()) for a,b in zip(refs,pred))/sum(len(a.split()) for a in refs)}\n    dump(output/\'predictions.json\',predictions);dump(output/\'metrics.json\',{\'model\':score([r[\'prediction\'] for r in predictions]),\'train_only_retrieval\':score(baseline),\'examples\':len(rows),\'official_benchmark\':False,\'linguistic_validation\':False})\n    dump(output/\'COMPLETED.json\',{\'steps\':trainer.state.global_step,\'best_checkpoint\':trainer.state.best_model_checkpoint,\'final_model\':\'final\',\'target_language\':\'Belgian LSFB; not France LSF\'})\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'cmd\',required=True)\n    a=sub.add_parser(\'extract\');a.add_argument(\'archive\');a.add_argument(\'output\');a.add_argument(\'--limit\',type=int,default=300)\n    a=sub.add_parser(\'split\');a.add_argument(\'input\');a.add_argument(\'output\');a.add_argument(\'--allow-weak\',action=\'store_true\')\n    a=sub.add_parser(\'train\');a.add_argument(\'data\');a.add_argument(\'output\');a.add_argument(\'--steps\',type=int,default=30);a.add_argument(\'--base\',default=\'google/mt5-small\');a.add_argument(\'--revision\');a.add_argument(\'--max-length\',type=int,default=96);a.add_argument(\'--eval-every\',type=int,default=0)\n    a=p.parse_args()\n    if a.cmd==\'extract\':print(extract(a.archive,a.output,a.limit))\n    elif a.cmd==\'split\':print(split(a.input,a.output,a.allow_weak))\n    else:train(a.data,a.output,a.steps,a.base,a.revision,a.max_length,a.eval_every)\n',encoding="utf-8")


## Train and select by validation
Identical 111/12/129 signer-disjoint examples as the mT5 comparison. T5 uses a 128-token limit to preserve every example under its tokenizer. Checkpoints save every 50 steps directly to Drive. Training resumes only complete checkpoints with a matching configuration. A 300-step model versus a 100-step model is an exploratory configuration comparison, not an equal-budget model benchmark.


In [ ]:
subprocess.run([sys.executable,str(SCRIPT),'train',str(DRIVE/'splits'),str(RUN),'--base',BASE_MODEL,'--revision',MODEL_REVISION,'--steps',str(TRAIN_STEPS),'--max-length','128','--eval-every','50'],check=True)
print((RUN/'metrics.json').read_text())
for example in json.loads((RUN/'predictions.json').read_text())[:10]:print(example)
print('Persistent model/tokenizer:',RUN/'final')
